# Model Intercomparison Demo
This notebook shows how to combine saved artifacts produced by the CLI for multiple models.

Updated per-model folder contents (standard naming patterns):
- `energy_spectra/`:
  - `*_spectrum_ens*.npz` (energy spectra per variable/level)
  - `lsd_2d_metrics_averaged_*.csv`, `lsd_2d_metrics_per_time_*.csv`
- `deterministic/`:
  - `deterministic_metrics*.csv` (may include `_averaged_`, `_initYYYY...`, `_lead000h-024h_`, etc.)
  - `deterministic_metrics_standardized*.csv`
- `ets/`:
  - `ets_metrics*.csv`
- `probabilistic/`:
  - `crps_summary*.csv` (CRPS summaries)
  - `pit_hist_*.npz` (PIT histograms)
  - `crps_map_*.npz` (CRPS spatial maps)
  - `probabilistic_metrics_spatial.nc`, `probabilistic_metrics_temporal.nc` (WBX aggregates, optional)
  - Optional WBX CSVs: `spread_skill_ratio.csv`, `crps_ensemble.csv`
- `vertical_profiles/`:
  - `*_pl_nmae_combined.npz` (per-variable NMAE lat-band curves)
  - Intercomparison adds `*_pl_nmae_combined_summary.csv`
- `histograms/`:
  - `hist_*_latbands_combined_ens*.npz` (lat-band distribution counts)
- `wd_kde/`:
  - `wd_kde_*combined_ens*.npz` (wind-direction KDE per lat band)
  - `wd_kde_wasserstein_averaged_*.csv`
- `maps/`:
  - `map_*_ens*.npz` (2D map fields; may include level / init / lead tokens)

All filenames embed standardized tokens produced by `helpers.build_output_filename`, notably:
- Time ranges: `initYYYYMMDDHH-YYYYMMDDHH`, `lead000h-024h`
- Ensemble: `_ens0`, `_ensmean`, or `_ensnone`

Below we configure model directories, run intercomparison modules, and inspect combined outputs.

## 1. Configuration

Edit the lists below to point to the per-model evaluation output folders you want to intercompare. Optionally provide human-readable labels (must match length of `model_dirs`). If `labels` is left empty or length mismatch occurs, folder basenames are used.

In [ ]:
from pathlib import Path

import pandas as pd

from swissclim_evaluations import intercompare

# --- 1. Configuration -------------------------------------------------------
# Provide paths to per-model evaluation output folders (each contains
# subdirectories like deterministic/, probabilistic/, energy_spectra/, etc.)
# Example assumption: you ran the main CLI with --out-dir output/modelA, output/modelB
# so that each model directory sits under project_root / "output".

# Resolve project root as parent of this notebooks/ folder (robust if executed from notebooks dir)
project_root = Path(__file__).resolve().parents[1]

# EDIT THESE to point to your real model output folders
model_dirs = [
    (project_root / "output/modelA").resolve(),
    (project_root / "output/modelB").resolve(),
    # Add more paths as needed
]

# Optional human-readable labels (must match number of model_dirs). If mismatch, folder names used.
labels = [
    "Model A",
    "Model B",
]

# Output root for combined intercomparison artifacts (created if missing)
# Can be relative to project root or absolute.
cfg_output = "output/intercomparison_notebook"
out_root = (
    (project_root / cfg_output).resolve() if not cfg_output.startswith("/") else Path(cfg_output)
)
out_root.mkdir(parents=True, exist_ok=True)

# Sub-modules to run: choose any subset of: spectra, hist, kde, maps, metrics, prob, vprof
modules = ["spectra", "hist", "kde", "maps", "metrics", "prob", "vprof"]

# Limit number of map / CRPS map panels to avoid huge output sizes
max_map_panels = 4  # maps
max_crps_map_panels = 4  # probabilistic CRPS maps

model_dirs

## 1a. Auto-Discover Model Output Folders
The helper below scans the `output/` directory (relative to project root) for sub-folders that contain at least one expected evaluation subdirectory (e.g. `deterministic`, `probabilistic`, `energy_spectra`). You can optionally use the discovered list to replace the manual `model_dirs` definition above.

In [ ]:
EXPECTED_SUBDIRS = {
    "deterministic",
    "probabilistic",
    "energy_spectra",
    "ets",
    "histograms",
    "wd_kde",
    "maps",
}


def discover_model_dirs(base: Path, min_hits: int = 1) -> list[Path]:
    """Return candidate model directories under base that contain at least `min_hits` expected subfolders.
    A candidate is any immediate child directory of `base` meeting the condition.
    """
    out: list[Path] = []
    if not base.is_dir():
        return out
    for child in sorted(p for p in base.iterdir() if p.is_dir()):
        subdirs = {d.name for d in child.iterdir() if d.is_dir()}
        hits = len(EXPECTED_SUBDIRS & subdirs)
        if hits >= min_hits:
            out.append(child.resolve())
    return out


# Discover under project_root / 'output'
auto_model_dirs = discover_model_dirs(project_root / "output", min_hits=1)
print("Discovered model directories:")
for p in auto_model_dirs:
    print(" -", p)

# To use discovered directories instead of manual list above, uncomment:
# model_dirs = auto_model_dirs
# labels = [p.name for p in model_dirs]

len(auto_model_dirs)

## 2. Sanity Checks

We verify that each provided model directory exists and has at least one expected sub-folder. Warnings are printed for missing folders so you can correct paths before running the full intercomparison.

In [ ]:
for p in model_dirs:
    if not p.exists():
        print(f"WARNING: Missing model directory: {p}")
    else:
        subdirs = [d.name for d in p.iterdir() if d.is_dir()]
        print(f"Found {p} with subdirs: {subdirs}")

## 3. Run Intercomparison Modules
The functions below discover common files across all provided models **using glob patterns** that match the standardized naming conventions. No manual filename editing should be needed when new metrics are added, as long as they follow the `build_output_filename` schema.

Examples of discovered patterns:
- Spectra: `energy_spectra/*_spectrum_ens*.npz`
- Histograms: `histograms/hist_*latbands_combined_ens*.npz`
- Wind-direction KDE: `wd_kde/wd_kde_*combined_ens*.npz`
- Maps: `maps/map_*.npz`
- Deterministic metrics: `deterministic/deterministic_metrics*.csv`
- ETS metrics: `ets/ets_metrics*.csv`
- Probabilistic CRPS: `probabilistic/crps_summary*.csv`
- PIT histograms: `probabilistic/pit_hist_*.npz`
- CRPS maps: `probabilistic/crps_map_*.npz`
- Vertical profiles: `vertical_profiles/*_pl_nmae_combined.npz`

Each intercomparison function produces combined CSVs or overlay figures with suffix `_compare.png` (for plots) or `_combined.csv` (for tabular merges).

In [ ]:
# Derive labels if length mismatch
if not labels or len(labels) != len(model_dirs):
    labels = [p.name for p in model_dirs]
print("Using labels:", labels)

# Execute selected intercomparison modules
if "spectra" in modules:
    intercompare.intercompare_energy_spectra(model_dirs, labels, out_root)
if "hist" in modules:
    intercompare.intercompare_histograms(model_dirs, labels, out_root)
if "kde" in modules:
    intercompare.intercompare_wd_kde(model_dirs, labels, out_root)
if "maps" in modules:
    intercompare.intercompare_maps(model_dirs, labels, out_root, max_panels=max_map_panels)
if "metrics" in modules:
    intercompare.intercompare_metrics_csv(model_dirs, labels, out_root)
if "prob" in modules:
    intercompare.intercompare_probabilistic(
        model_dirs, labels, out_root, max_crps_map_panels=max_crps_map_panels
    )
if "vprof" in modules:
    intercompare.intercompare_vertical_profiles(model_dirs, labels, out_root)

print("Intercomparison complete. Artifacts written to", out_root.resolve())

## 4. Quick Look at Combined Metrics

If deterministic / probabilistic metrics were combined, load and display a few of them inline for inspection.

## 4. Deterministic Metrics Preview
If the deterministic intercomparison module ran, you should have:
- `deterministic/metrics_combined.csv`
- `deterministic/metrics_standardized_combined.csv`

The snippet below loads and previews them (showing the first few rows and available metric columns).

In [ ]:
det_root = out_root / "deterministic"
metrics_combined = det_root / "metrics_combined.csv"
metrics_std_combined = det_root / "metrics_standardized_combined.csv"

if metrics_combined.exists():
    df_det = pd.read_csv(metrics_combined)
    print("Deterministic combined shape:", df_det.shape)
    print("Columns (first 15):", df_det.columns.tolist()[:15])
    display(df_det.head())
else:
    print("Deterministic combined metrics CSV not found.")

if metrics_std_combined.exists():
    df_det_std = pd.read_csv(metrics_std_combined)
    print("Standardized deterministic combined shape:", df_det_std.shape)
    print("Columns (first 15):", df_det_std.columns.tolist()[:15])
    display(df_det_std.head())
else:
    print("Standardized deterministic combined CSV not found.")

## 5. ETS Metrics Quick Plots
If `ets/ets_metrics_combined.csv` exists, we produce quick bar plots for selected categorical thresholds (if columns present). The combined CSV contains stacked model rows; we pivot by `model` for each metric variant.

In [ ]:
import matplotlib.pyplot as plt

ets_root = out_root / "ets"
ets_combined = ets_root / "ets_metrics_combined.csv"

if ets_combined.exists():
    df_ets = pd.read_csv(ets_combined)
    print("ETS combined shape:", df_ets.shape)
    # Identify possible metric columns (exclude obvious id columns)
    id_like = {
        "model",
        "variable",
        "threshold",
        "init_start",
        "init_end",
        "lead_start",
        "lead_end",
    }
    metric_cols = [c for c in df_ets.columns if c not in id_like and df_ets[c].dtype != object]
    print("Metric columns detected:", metric_cols[:10])
    # If a threshold column exists, we can group by threshold; else just by variable
    has_threshold = "threshold" in df_ets.columns
    group_dim = (
        "threshold" if has_threshold else ("variable" if "variable" in df_ets.columns else None)
    )
    # Limit number of metrics plotted for brevity
    for metric in metric_cols[:5]:
        plt.figure(figsize=(10, 5))
        if group_dim:
            # Average across other dimensions except group_dim and model
            agg_cols = [c for c in [group_dim, "model", metric] if c in df_ets.columns]
            tmp = (
                df_ets[agg_cols]
                .groupby([group_dim, "model"], as_index=False)
                .mean(numeric_only=True)
            )
            pivot = tmp.pivot(index=group_dim, columns="model", values=metric)
            pivot.plot(kind="bar")
            plt.ylabel(metric)
            plt.title(f"{metric} by {group_dim} and model")
            plt.tight_layout()
        else:
            tmp = df_ets.groupby("model", as_index=False)[metric].mean(numeric_only=True)
            plt.bar(tmp["model"], tmp[metric])
            plt.ylabel(metric)
            plt.title(f"{metric} (mean across rows)")
            plt.xticks(rotation=45)
            plt.tight_layout()
        out_png = ets_root / f"{metric}_quick_compare.png"
        plt.savefig(out_png, bbox_inches="tight", dpi=160)
        plt.close()
        print("Saved", out_png)
else:
    print("No ets_metrics_combined.csv found; skip ETS plotting.")

In [ ]:
# Probabilistic combined metrics overview
prob_out = out_root / "probabilistic"
prob_out.mkdir(exist_ok=True, parents=True)

summary_crps = prob_out / "crps_summary_combined.csv"
summary_spatial = prob_out / "spatial_metrics_combined.csv"
summary_temporal = prob_out / "temporal_metrics_combined.csv"
print("CRPS summary combined exists:", summary_crps.exists())
print("Spatial metrics combined exists:", summary_spatial.exists())
print("Temporal metrics combined exists:", summary_temporal.exists())

if summary_crps.exists():
    df_crps = pd.read_csv(summary_crps)
    display(df_crps.head())
else:
    print(
        "CRPS combined summary not found (maybe no probabilistic module run or no matching files)."
    )

## 6. Vertical Profile (NMAE) Overlays
If the vertical profile intercomparison ran (module `vprof`), you should see per-variable comparison images and summary CSVs in `vertical_profiles/` with naming patterns:
- `*_pl_nmae_combined_compare.png` — overlay plot (north & south bands)
- `*_pl_nmae_combined_summary.csv` — tidy per-band average NMAE values

The cell below lists available artifacts and builds a compact summary table of mean NMAE per model & variable.

In [ ]:
vp_root = out_root / "vertical_profiles"

pngs = sorted(vp_root.glob("*_pl_nmae_combined_compare.png"))
summaries = sorted(vp_root.glob("*_pl_nmae_combined_summary.csv"))
print(f"Found {len(pngs)} comparison PNG(s) and {len(summaries)} summary CSV(s).")
for p in pngs[:10]:  # list first 10 to avoid flooding output
    print("IMG:", p.name)
if len(pngs) > 10:
    print("... (truncated list)")
for s in summaries[:10]:
    print("CSV:", s.name)
if len(summaries) > 10:
    print("... (truncated list)")

# Build combined tidy DataFrame from all summaries if present
import pandas as _pd

rows = []
for csv in summaries:
    try:
        df = _pd.read_csv(csv)
    except Exception:
        continue
    # Expect columns: variable, band_index, hemisphere, model, value, metric
    expected = {"variable", "band_index", "hemisphere", "model", "value"}
    if not expected.issubset(set(df.columns)):
        continue
    rows.append(df)

if rows:
    combined_vprof = _pd.concat(rows, ignore_index=True)
    # Aggregate: mean NMAE per variable & model (averaging over bands+hemispheres)
    agg = combined_vprof.groupby(["variable", "model"], as_index=False)["value"].mean()
    pivot = agg.pivot(index="variable", columns="model", values="value")
    print("Mean NMAE (%) by variable and model (lower is better):")
    display(pivot)
else:
    print("No valid vertical profile summary CSVs to aggregate.")

## 7. Exploring Combined Spatial / Temporal Probabilistic Metrics (WBX)
If WeatherBenchX WBX artifacts were produced, you can further slice / visualize them here.

In [ ]:
# Explore spatial / temporal WBX aggregates if present
spatial_comb = out_root / "probabilistic" / "spatial_metrics_combined.csv"
temporal_comb = out_root / "probabilistic" / "temporal_metrics_combined.csv"

if spatial_comb.exists():
    df_spatial = pd.read_csv(spatial_comb)
    print("Spatial metrics columns:", df_spatial.columns.tolist()[:15], "...")
    display(df_spatial.head())
else:
    print("No combined spatial metrics CSV found.")

if temporal_comb.exists():
    df_temporal = pd.read_csv(temporal_comb)
    print("Temporal metrics columns:", df_temporal.columns.tolist()[:15], "...")
    display(df_temporal.head())
else:
    print("No combined temporal metrics CSV found.")